In [1]:
!pip install -q gradio pypdf peft transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 5.2 MB/s eta 0:00:00


In [4]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [6]:
MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/capstone/models/cuad-clause-extractor-v1"

BASE_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"

print(MODEL_PATH)

/content/drive/MyDrive/Colab Notebooks/capstone/models/cuad-clause-extractor-v1


In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded")

Tokenizer loaded


In [8]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto"
)

print("Base model loaded")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base model loaded


In [8]:
!pip uninstall -y torchao
!pip install -U torchao peft transformers accelerate

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 79.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1


In [9]:
model = PeftModel.from_pretrained(
    base_model,
    MODEL_PATH
)

model.eval()

print("CUAD fine-tuned model ready")

CUAD fine-tuned model ready


In [11]:
def extract_clauses(contract):

    prompt = f"""
You are a legal contract analysis assistant.
Extract important clauses and return JSON.

Extract:
- governing_law
- termination
- indemnification
- liability_cap
- non_compete
- exclusivity

Contract:

{contract}

Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)


    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False
        )


    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # remove prompt text
    result = result.replace(prompt, "")

    return result

In [12]:
test_contract = """
This Agreement shall be governed by the laws of the State of New York.

Either party may terminate this Agreement after thirty days written notice.

The Company shall indemnify the other party against all claims.
"""

result = extract_clauses(test_contract)

print(result)

{
  "governing_law": "New York",
  "termination": "30 days written notice",
  "indemnification": "Neither party shall be liable for any claims against the other party.",
  "liability_cap": "30000000000000000000000000000000000000000000000000000000000000000000


In [13]:
!pip install -q gradio

In [14]:
def analyze_contract(text):

    if len(text.strip()) == 0:
        return "Please enter contract text"

    result = extract_clauses(text)

    return result

In [15]:
import gradio as gr


demo = gr.Interface(
    fn=analyze_contract,

    inputs=gr.Textbox(
        label="Paste Contract Text",
        lines=15,
        placeholder="Enter legal contract text here..."
    ),

    outputs=gr.Textbox(
        label="Extracted Clauses",
        lines=15
    ),

    title="CUAD Legal Clause Extractor",

    description="Fine-tuned SmolLM2-360M + LoRA model for legal clause extraction"
)


demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f4dad16f160b5f3a55.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
